In [ ]:
# Google Colab : télécharger les données de cette séance / download this session's data
import os, urllib.request
BASE = "https://raw.githubusercontent.com/Marzoug-Nabil/Nabil/main/academic/teaching/fsci6222/materials/tp2/"
for f in ['diabete.csv']:
    os.makedirs('data', exist_ok=True)
    for dst in (f, os.path.join('data', f)):
        if not os.path.exists(dst):
            urllib.request.urlretrieve(BASE + f, dst)


# FSCI 6222 -- Analyse de donnees
# Travail pratique no 2 -- Analyse predictive du diabete

**Universite de Moncton -- Faculte des sciences -- Ete 2026**

---

- **Date de distribution :** Mardi 2 juin 2026
- **Date de remise :** jeudi 11 juin 2026, avant 23h00
- **Mode de remise :** sur CLIC, format `TP2_NomPrenom.zip`
- **Ponderation :** 15% de la note finale
- **Total :** 50 points

**Identification :**

- Nom : 
- Prenom : 
- Numero d'etudiant : 

---

**Contexte.** Le diabete de type 2 est l'une des maladies chroniques les plus repandues. Sa detection precoce, grace a l'analyse des donnees cliniques, peut orienter les patientes vers une prise en charge adaptee. Dans ce TP, vous allez analyser un jeu de donnees cliniques anonymisees et construire un modele predictif simple du diabete.

**Consignes :**

- Tous les exercices utilisent le fichier `diabete.csv` (fourni avec ce notebook).
- Le code doit etre commente, et les resultats interpretes en cellules Markdown.
- Travail individuel. Le plagiat sera sanctionne.
- Penalite de retard : -10% par tranche de 24h, note de 0 apres 5 jours.

**Description du dataset `diabete.csv` :** donnees cliniques anonymisees d'environ 400 patientes. Colonnes : `id_patient`, `grossesses`, `glycemie` (mg/dL), `pression_arterielle` (mmHg), `pli_cutane` (mm), `insuline` (mu U/mL), `imc` (kg/m2), `heredite` (score d'antecedents familiaux), `age`, `diabete` (1 = diabetique, 0 = non). Le fichier contient des valeurs manquantes et des doublons (volontairement).


## Importations

Executez d'abord cette cellule. Vous pouvez ajouter d'autres importations au besoin.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import (r2_score, mean_squared_error,
                              accuracy_score, confusion_matrix)

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
print('Environnement pret.')

---

# Section 1 -- Exploration et probabilites (10 points)


## 1.1 Chargement et nettoyage (3 points)


**Q1.** Chargez `diabete.csv` dans un DataFrame `df`. Affichez ses dimensions, les 5 premieres lignes et les types de donnees. *(1 pt)*


In [ ]:
# Votre code ici


**Q2.** Identifiez le nombre de valeurs manquantes par colonne et le nombre de doublons. *(1 pt)*


In [ ]:
# Votre code ici


**Q3.** Creez un DataFrame nettoye `df_clean` : supprimez les doublons et remplissez les valeurs manquantes par la **mediane** de la colonne concernee. Verifiez qu'il ne reste aucune valeur manquante. *(1 pt)*


In [ ]:
# Votre code ici


## 1.2 Distribution de la glycemie (4 points)


**Q4.** Tracez un **histogramme** de la colonne `glycemie`. Calculez sa moyenne et son ecart-type. *(1,5 pt)*


In [ ]:
# Votre code ici


**Q5.** En supposant que `glycemie` suit une loi **normale** de parametres egaux a la moyenne et l'ecart-type calcules, utilisez `scipy.stats.norm` pour calculer la **probabilite qu'une patiente tiree au hasard ait une glycemie superieure a 140 mg/dL** (seuil clinique d'alerte). *(1,5 pt)*


In [ ]:
# Votre code ici


> **Rappel methodologique : le test de Shapiro-Wilk.** Ce test verifie si un echantillon provient d'une distribution normale. On utilise `scipy.stats.shapiro(donnees)`, qui retourne deux valeurs : la **statistique de test** et la **p-value**.
> 
> **Regle de decision (seuil 5%) :**
> - Si **p-value > 0,05** : on **ne rejette pas** la normalite (la distribution peut etre consideree normale).
> - Si **p-value <= 0,05** : on **rejette** la normalite.

**Q6.** Effectuez un test de **Shapiro-Wilk** sur `glycemie`. Au seuil de 5%, peut-on conclure que la distribution est normale ? Justifiez votre reponse a partir de la p-value. *(1 pt)*


In [ ]:
# Votre code ici


*Interpretation :*


## 1.3 Probabilites discretes (3 points)


**Q7.** Calculez la **proportion empirique** de patientes diabetiques (`diabete = 1`) dans le dataset. Notez cette proportion `p`. *(1 pt)*


In [ ]:
# Votre code ici


**Q8.** En supposant que le statut diabetique d'une patiente suit une loi de Bernoulli de parametre `p`, et qu'on selectionne 10 patientes au hasard, utilisez `scipy.stats.binom` pour calculer la probabilite qu'**au moins 3 patientes** soient diabetiques. *(2 pt)*


In [ ]:
# Votre code ici


---

# Section 2 -- Correlation et regression (15 points)


## 2.1 Analyse des correlations (5 points)


**Q9.** Calculez la **matrice de correlation de Pearson** entre toutes les variables numeriques de `df_clean` (sauf `id_patient`). Affichez-la sous forme de **heatmap**. *(2 pt)*


In [ ]:
# Votre code ici


**Q10.** Identifiez les **trois variables les plus correlees** avec `diabete`. Donnez leurs coefficients et commentez leur sens (positif/negatif) du point de vue clinique. *(1,5 pt)*


In [ ]:
# Votre code ici


*Reponse :*


**Q11.** Pour la variable la plus correlee avec `diabete`, calculez egalement la correlation de **Spearman**. Comparez avec Pearson : la relation est-elle plutot lineaire ou monotone non lineaire ? *(1,5 pt)*


In [ ]:
# Votre code ici


*Interpretation :*


## 2.2 Regression lineaire (10 points)


**Q12.** Construisez une **regression lineaire simple** predisant `diabete` a partir de `glycemie`. Affichez la pente, l'ordonnee a l'origine et le R2. Interpretez la pente. *(2 pt)*


In [ ]:
# Votre code ici


*Interpretation de la pente :*


**Q13.** Construisez une **regression lineaire multiple** predisant `diabete` a partir de toutes les variables cliniques (`grossesses`, `glycemie`, `pression_arterielle`, `pli_cutane`, `insuline`, `imc`, `heredite`, `age`). *(2 pt)*


In [ ]:
# Votre code ici


**Q14.** Separez les donnees en jeu d'entrainement (80%) et jeu de test (20%) avec `random_state=42`. Reentrainez le modele multiple sur le jeu d'entrainement et evaluez le **R2** et le **RMSE** sur le jeu de test. *(2,5 pt)*


In [ ]:
# Votre code ici


**Q15.** Tracez le **diagramme des residus** (residus en fonction des valeurs predites) pour le modele multiple. Que pouvez-vous conclure de ce diagramme ? *(2 pt)*


In [ ]:
# Votre code ici


*Interpretation :*


**Q16.** Comparez le R2 de la regression simple (Q12) avec le R2 test de la regression multiple (Q14). Justifiez la difference : quelles variables apportent une information supplementaire ? *(1,5 pt)*


*Reponse :*


---

# Section 3 -- Reduction de dimensionnalite (10 points)

Pour cette section, utilisez les huit variables cliniques (toutes sauf `id_patient` et `diabete`).


## 3.1 Preparation des donnees (2 points)


**Q17.** Pourquoi est-il **indispensable de standardiser** les variables avant d'appliquer une PCA ou un t-SNE ? Expliquez en une ou deux phrases. *(1 pt)*


*Reponse :*


**Q18.** Appliquez `StandardScaler` de scikit-learn aux huit variables cliniques de `df_clean` et stockez le resultat dans une variable `X_scaled`. *(1 pt)*


In [ ]:
# Votre code ici


## 3.2 PCA (5 points)


**Q19.** Appliquez une **PCA** sur `X_scaled` en gardant les 2 premieres composantes principales. Affichez la **variance expliquee** par chacune des deux composantes, et le total cumule. *(2 pt)*


In [ ]:
# Votre code ici


**Q20.** Tracez un **nuage de points** des patientes dans le nouvel espace (PC1 sur l'axe x, PC2 sur l'axe y), colore selon `diabete` (par exemple bleu = non, rouge = oui). Ajoutez une legende et un titre. *(2 pt)*


In [ ]:
# Votre code ici


**Q21.** Commentez le resultat : observez-vous des **regroupements visibles** entre patientes diabetiques et non diabetiques dans l'espace PCA ? *(1 pt)*


*Reponse :*


## 3.3 t-SNE (3 points)


**Q22.** Appliquez un **t-SNE** sur `X_scaled` avec `n_components=2`, `perplexity=30` et `random_state=42`. Tracez le nuage de points obtenu, colore selon `diabete`. *(2 pt)*


In [ ]:
# Votre code ici


**Q23.** Comparez visuellement le resultat t-SNE avec celui de la PCA (Q20). Lequel des deux semble mieux separer les patientes selon leur statut diabetique ? *(1 pt)*


*Reponse :*


---

# Section 4 -- Apprentissage automatique (15 points)

L'objectif est de **predire** si une patiente est diabetique ou non, a partir de ses donnees cliniques.


## 4.1 Preparation du probleme de classification (3 points)


**Q24.** Affichez la distribution de la cible `diabete` (nombre de patientes dans chaque classe). La distribution est-elle equilibree ? *(1,5 pt)*


In [ ]:
# Votre code ici


*Reponse :*


**Q25.** Separez les donnees standardisees (`X_scaled`) et la cible `diabete` en jeu d'entrainement (80%) et jeu de test (20%) avec `random_state=42`. *(1,5 pt)*


In [ ]:
# Votre code ici


## 4.2 Classification : KNN vs Arbre de decision (8 points)


**Q26.** Entrainez un classificateur **KNN** avec `n_neighbors=5` sur le jeu d'entrainement. Calculez son **accuracy** sur le jeu de test. *(2 pt)*


In [ ]:
# Votre code ici


**Q27.** Entrainez un **arbre de decision** (`DecisionTreeClassifier`) avec `max_depth=4` et `random_state=42`. Calculez son **accuracy** sur le jeu de test. *(2 pt)*


In [ ]:
# Votre code ici


**Q28.** Affichez la **matrice de confusion** de chaque modele sur le jeu de test. *(2 pt)*


In [ ]:
# Votre code ici


**Q29.** Comparez les deux modeles : lequel obtient la meilleure accuracy ? Pour un test medical de depistage du diabete, est-il plus grave de manquer une vraie diabetique (faux negatif) ou de declarer diabetique une personne saine (faux positif) ? Justifiez votre choix de modele dans ce contexte. *(2 pt)*


*Reponse :*


## 4.3 Clustering : K-Means (4 points)


**Q30.** Appliquez l'algorithme **K-Means** avec `n_clusters=2` et `random_state=42` sur `X_scaled`. Affichez la **taille de chaque cluster**. *(2 pt)*


In [ ]:
# Votre code ici


**Q31.** Comparez les clusters obtenus avec la variable `diabete` en utilisant un `pd.crosstab`. Les clusters correspondent-ils aux deux groupes (diabetique/non) ? Expliquez. *(2 pt)*


In [ ]:
# Votre code ici


*Interpretation :*


---

**Fin du TP2.** Verifiez que toutes vos cellules s'executent sans erreur, puis compressez votre archive `TP2_NomPrenom.zip` (avec ce notebook et `diabete.csv`) et remettez-la sur CLIC avant **jeudi 11 juin 2026, 23h00**.
